In [1]:
from utilities.utils import SSPModelForCalibration, HelperFunctions
from utilities.diff_reports_v2 import DiffReportUtils
from sisepuede.manager.sisepuede_examples import SISEPUEDEExamples
import pandas as pd
import warnings
import os
warnings.filterwarnings("ignore")

/home/tony-ubuntu/anaconda3/envs/opt_env/lib/python3.11/site-packages/munch/__init__.py:24: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Initialize helper functions
helper_functions = HelperFunctions()

In [4]:
# Define paths
SCRIPTS_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPTS_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, 'data')
OUTPUT_DIR_PATH = os.path.join(ROOT_DIR_PATH, 'output')
MISC_DIR_PATH = os.path.join(SCRIPTS_DIR_PATH, 'misc')
DUMMY_DIR_PATH = os.path.join(MISC_DIR_PATH, 'dummy')
SECTORAL_REPORT_MAPPING_DIR_PATH = os.path.join(MISC_DIR_PATH, 'sectoral_report_mapping')
SECTORAL_REPORTS_DIR_PATH = os.path.join(MISC_DIR_PATH, 'sectoral_reports')

In [5]:
# Define parameters
iso_alpha_3 = 'UGA'
region = 'uganda'
energy_model_flag = False
run_id = '20250723221813'
emission_targets_file = "emission_targets_uganda.csv"
ssp_edgar_cw_file_name = 'sisepuede_edgar_active_crosswalk.csv'
RUN_DIR_PATH = os.path.join(OUTPUT_DIR_PATH, region, run_id)
INPUT_DATA_PATH = os.path.join(DATA_DIR_PATH, "new_uganda_inputs.csv")

## Here we create the reports and we compare them to check how much improvement we have in each subsector with the calibration

In [6]:
# Load the output dataset
if energy_model_flag:
    output_df_path = os.path.join(DUMMY_DIR_PATH, f'ssp_{region}_output_dummy_energy.csv')
else:
    output_df_path = os.path.join(DUMMY_DIR_PATH, f'ssp_{region}_output_dummy.csv')

output_df = pd.read_csv(output_df_path)
output_df.head()

,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,area_agrc_crops_other_woody_perennial,area_agrc_crops_pulses,...,yield_agrc_fruits_tonne,yield_agrc_herbs_and_other_perennial_crops_tonne,yield_agrc_nuts_tonne,yield_agrc_other_annual_tonne,yield_agrc_other_woody_perennial_tonne,yield_agrc_pulses_tonne,yield_agrc_rice_tonne,yield_agrc_sugar_cane_tonne,yield_agrc_tubers_tonne,yield_agrc_vegetables_and_vines_tonne
0,0,747901.028486,2.386299e+06,183833.867477,1.333131e+06,595.329750,605228.344795,3.433943e+06,4242.254753,1.090824e+06,...,9.829842e+06,1264.067389,461200.281192,3.374662e+06,1558.604070,855742.438638,371284.928361,8.144525e+06,8.463351e+06,2.839233e+06
1,1,746180.919732,2.380811e+06,183411.065217,1.330065e+06,593.960543,603836.371078,3.426045e+06,4232.497928,1.088315e+06,...,9.589512e+06,1289.597171,432449.975084,3.372849e+06,1555.019391,874816.334236,379914.032975,8.196196e+06,8.657214e+06,2.881371e+06
2,2,745855.348591,2.379772e+06,183331.039920,1.329485e+06,593.701388,603572.907231,3.424550e+06,4230.651219,1.087840e+06,...,9.555432e+06,1268.550416,465394.440892,3.423848e+06,1554.340889,879399.053014,390826.724560,8.171843e+06,9.876628e+06,2.834867e+06
3,3,746831.986263,2.382888e+06,183571.097192,1.331226e+06,594.478792,604363.237474,3.429034e+06,4236.190917,1.089264e+06,...,9.549210e+06,1268.638786,450148.390655,4.942354e+06,1556.376205,876612.253332,398412.478937,8.131891e+06,1.065485e+07,2.843764e+06
4,4,748702.931966,2.388858e+06,184030.975132,1.334561e+06,595.968066,605877.273861,3.437624e+06,4246.803321,1.091993e+06,...,9.634575e+06,1294.481423,277564.514174,5.253808e+06,1560.275189,873391.596641,400599.954899,8.252278e+06,9.158196e+06,2.865777e+06


In [7]:
edgar_ssp_cw_path = os.path.join(SECTORAL_REPORT_MAPPING_DIR_PATH, ssp_edgar_cw_file_name)

dru = DiffReportUtils(iso_alpha_3, edgar_ssp_cw_path, SECTORAL_REPORTS_DIR_PATH, energy_model_flag)

In [8]:

edgar_emission_targets_path = os.path.join(SECTORAL_REPORT_MAPPING_DIR_PATH, emission_targets_file)
edgar_emission_df = dru.get_edgar_region_df(edgar_emission_targets_path)
    
edgar_emission_df.head()

,ids,iso_alpha_3,edgar_emission,year
0,1:lvst-lsmm:ch4,UGA,9.972244,2015
1,1:lvst-lsmm:ch4,UGA,9.972244,2015
2,3:lsmm:n2o,UGA,0.299059,2015
3,4:agrc:co2,UGA,0.000000,2015
4,5:agrc:ch4,UGA,0.576887,2015


In [9]:
report_dict = dru.run_report_generator(edgar_emission_df, output_df)

sectoral_emission_report = report_dict['sectoral_emission_report']
subsector_emission_report = report_dict['subsector_emission_report']
model_failed_flag = report_dict['model_failed_flag']

In [10]:
og_report = sectoral_emission_report.copy()
og_report.head()

,ids,subsector,ssp_emission,iso_alpha_3,edgar_emission,year,edgar_emission_epsilon,rel_error,squared_diff,direct_weight,norm_weight,log_weight
0,16:inen:co2,inen,30.331529,UGA,1.663096,2015,1.663097,17.237980,821.879005,1.663097,0.023887,0.979489
1,7:scoe:co2,scoe,0.000000,UGA,3.489266,2015,3.489267,-1.000000,12.174987,3.489267,0.050117,1.501689
2,19:trns:co2,trns,5.389002,UGA,3.903503,2015,3.903504,0.380555,2.206701,3.903504,0.056067,1.589950
9,1:lvst-lsmm:ch4,lvst,62.373950,UGA,19.944488,2015,19.944489,2.127378,1800.259200,19.944489,0.286467,3.041876
10,22:ippu:co2,ippu,1.954773,UGA,1.765612,2015,1.765613,0.107136,0.035782,1.765613,0.025360,1.017262


In [11]:
og_report

,ids,subsector,ssp_emission,iso_alpha_3,edgar_emission,year,edgar_emission_epsilon,rel_error,squared_diff,direct_weight,norm_weight,log_weight
0,16:inen:co2,inen,30.331529,UGA,1.663096,2015,1.663097,17.237980,8.218790e+02,1.663097,0.023887,0.979489
1,7:scoe:co2,scoe,0.000000,UGA,3.489266,2015,3.489267,-1.000000,1.217499e+01,3.489267,0.050117,1.501689
2,19:trns:co2,trns,5.389002,UGA,3.903503,2015,3.903504,0.380555,2.206701e+00,3.903504,0.056067,1.589950
9,1:lvst-lsmm:ch4,lvst,62.373950,UGA,19.944488,2015,19.944489,2.127378,1.800259e+03,19.944489,0.286467,3.041876
10,22:ippu:co2,ippu,1.954773,UGA,1.765612,2015,1.765613,0.107136,3.578159e-02,1.765613,0.025360,1.017262
11,23:ippu:ch4,ippu,0.006587,UGA,0.000000,2015,0.000001,6586.248325,4.337867e-05,0.000001,0.000000,0.000000
12,24:ippu:n2o,ippu,0.541484,UGA,0.534607,2015,0.534608,0.012862,4.728029e-05,0.534608,0.007679,0.428275
13,3:lsmm:n2o,lsmm,6.355724,UGA,0.299059,2015,0.299060,20.252354,3.668318e+01,0.299060,0.004295,0.261640
14,4:agrc:co2,agrc,0.472549,UGA,0.000000,2015,0.000001,472548.279207,2.233019e-01,0.000001,0.000000,0.000000
15,50:ippu:hfcs,ippu,0.000000,UGA,0.000000,2015,0.000001,-1.000000,1.000000e-12,0.000001,0.000000,0.000000


In [12]:
# Sort by ids and edgar_class
og_report = og_report.sort_values(by=["ids"]).reset_index(drop=True)

# Filter out the columns we need
og_report = og_report[['ids', 'ssp_emission', 'rel_error']]
og_report

,ids,ssp_emission,rel_error
0,16:inen:co2,30.331529,17.237980
1,19:trns:co2,5.389002,0.380555
2,1:lvst-lsmm:ch4,62.373950,2.127378
3,22:ippu:co2,1.954773,0.107136
4,23:ippu:ch4,0.006587,6586.248325
5,24:ippu:n2o,0.541484,0.012862
6,3:lsmm:n2o,6.355724,20.252354
7,4:agrc:co2,0.472549,472548.279207
8,50:ippu:hfcs,0.000000,-1.000000
9,51:ippu:other_fcs,0.000000,-1.000000


In [13]:
opt_detailed_report_path = os.path.join(OUTPUT_DIR_PATH, region, run_id, f'best_detailed_diff_report_{run_id}.csv')
# opt_detailed_report_path = os.path.join(DUMMY_DIR_PATH, 'sectoral_emission_report_dummy.csv')
opt_report = pd.read_csv(opt_detailed_report_path)
opt_report.head()

,ids,subsector,ssp_emission,iso_alpha_3,edgar_emission,year,edgar_emission_epsilon,rel_error,squared_diff,direct_weight,norm_weight,log_weight
0,16:inen:co2,inen,7.345460,UGA,1.663096,2022,1.663097,3.416736,32.289253,1.663097,0.023887,0.979489
1,7:scoe:co2,scoe,0.000000,UGA,3.489266,2022,3.489267,-1.000000,12.174987,3.489267,0.050117,1.501689
2,19:trns:co2,trns,4.403192,UGA,3.903503,2022,3.903504,0.128010,0.249688,3.903504,0.056067,1.589950
3,1:lvst-lsmm:ch4,lvst,19.898780,UGA,19.944488,2022,19.944489,-0.002292,0.002089,19.944489,0.286467,3.041876
4,22:ippu:co2,ippu,1.460823,UGA,1.765612,2022,1.765613,-0.172626,0.092897,1.765613,0.025360,1.017262


In [14]:
# Sort the report by subsector and edgar_class so we don't mess up with the numeric_id
opt_report = opt_report.sort_values(by=['ids']).reset_index(drop=True)

# Filter out the columns we need
opt_report = opt_report[['ids', 'ssp_emission', 'edgar_emission_epsilon', 'norm_weight', 'rel_error']]
opt_report

,ids,ssp_emission,edgar_emission_epsilon,norm_weight,rel_error
0,16:inen:co2,7.345460,1.663097,0.023887,3.416736e+00
1,19:trns:co2,4.403192,3.903504,0.056067,1.280100e-01
2,1:lvst-lsmm:ch4,19.898780,19.944489,0.286467,-2.291814e-03
3,22:ippu:co2,1.460823,1.765613,0.025360,-1.726256e-01
4,23:ippu:ch4,0.013568,0.000001,0.000000,1.356687e+04
5,24:ippu:n2o,0.393869,0.534608,0.007679,-2.632568e-01
6,3:lsmm:n2o,3.182868,0.299060,0.004295,9.642916e+00
7,4:agrc:co2,0.498790,0.000001,0.000000,4.987890e+05
8,50:ippu:hfcs,0.000000,0.000001,0.000000,-1.000000e+00
9,51:ippu:other_fcs,0.000000,0.000001,0.000000,-1.000000e+00


In [15]:
merged_df = pd.merge(og_report, opt_report, how='outer', on=['ids'])
new_col_names = {
    'rel_error_x': 'rel_error_og',
    'rel_error_y': 'rel_error_opt',
    'edgar_emission_epsilon': 'edgar_emission_value',
    'norm_weight': 'emission_contribution_per',
    'ssp_emission_x': 'ssp_emission_value_og',
    'ssp_emission_y': 'ssp_emission_value_opt'
}

merged_df.rename(columns= new_col_names, inplace=True)

# Add weighted_rel_error column
merged_df['weighted_rel_error_opt'] = merged_df['rel_error_opt'] * merged_df['emission_contribution_per']
merged_df['weighted_rel_error_og'] = merged_df['rel_error_og'] * merged_df['emission_contribution_per']

merged_df.head()

,ids,ssp_emission_value_og,rel_error_og,ssp_emission_value_opt,edgar_emission_value,emission_contribution_per,rel_error_opt,weighted_rel_error_opt,weighted_rel_error_og
0,16:inen:co2,30.331529,17.237980,7.345460,1.663097,0.023887,3.416736,0.081617,0.411770
1,19:trns:co2,5.389002,0.380555,4.403192,3.903504,0.056067,0.128010,0.007177,0.021336
2,1:lvst-lsmm:ch4,62.373950,2.127378,19.898780,19.944489,0.286467,-0.002292,-0.000657,0.609423
3,22:ippu:co2,1.954773,0.107136,1.460823,1.765613,0.025360,-0.172626,-0.004378,0.002717
4,23:ippu:ch4,0.006587,6586.248325,0.013568,0.000001,0.000000,13566.866717,0.000000,0.000000


In [16]:
# Check for null values in the merged DataFrame
merged_df.isnull().sum()

ids                          0
ssp_emission_value_og        0
rel_error_og                 0
ssp_emission_value_opt       0
edgar_emission_value         0
emission_contribution_per    0
rel_error_opt                0
weighted_rel_error_opt       0
weighted_rel_error_og        0
dtype: int64

In [17]:
# Reorder the columns
merged_df = merged_df[['ids',
                       'ssp_emission_value_og',
                       'ssp_emission_value_opt', 
                       'edgar_emission_value', 
                       'emission_contribution_per', 
                       'rel_error_og', 
                       'rel_error_opt', 
                       'weighted_rel_error_og', 
                       'weighted_rel_error_opt', 
                       ]]
merged_df

,ids,ssp_emission_value_og,ssp_emission_value_opt,edgar_emission_value,emission_contribution_per,rel_error_og,rel_error_opt,weighted_rel_error_og,weighted_rel_error_opt
0,16:inen:co2,30.331529,7.345460,1.663097,0.023887,17.237980,3.416736e+00,0.411770,0.081617
1,19:trns:co2,5.389002,4.403192,3.903504,0.056067,0.380555,1.280100e-01,0.021336,0.007177
2,1:lvst-lsmm:ch4,62.373950,19.898780,19.944489,0.286467,2.127378,-2.291814e-03,0.609423,-0.000657
3,22:ippu:co2,1.954773,1.460823,1.765613,0.025360,0.107136,-1.726256e-01,0.002717,-0.004378
4,23:ippu:ch4,0.006587,0.013568,0.000001,0.000000,6586.248325,1.356687e+04,0.000000,0.000000
5,24:ippu:n2o,0.541484,0.393869,0.534608,0.007679,0.012862,-2.632568e-01,0.000099,-0.002021
6,3:lsmm:n2o,6.355724,3.182868,0.299060,0.004295,20.252354,9.642916e+00,0.086993,0.041421
7,4:agrc:co2,0.472549,0.498790,0.000001,0.000000,472548.279207,4.987890e+05,0.000000,0.000000
8,50:ippu:hfcs,0.000000,0.000000,0.000001,0.000000,-1.000000,-1.000000e+00,-0.000000,-0.000000
9,51:ippu:other_fcs,0.000000,0.000000,0.000001,0.000000,-1.000000,-1.000000e+00,-0.000000,-0.000000


In [18]:
# Calculate the sum of the absolute value of the weighted_rel_error_opt and weighted_rel_error_og
total_w_rel_error_opt = merged_df['weighted_rel_error_opt'].abs().sum()
total_w_rel_error_og = merged_df['weighted_rel_error_og'].abs().sum()

# Create a dataframe to store the total weighted relative error
total_w_rel_error_df = pd.DataFrame({
    'total_w_rel_error_og': [total_w_rel_error_og],
    'total_w_rel_error_opt': [total_w_rel_error_opt]
    
})

total_w_rel_error_df

,total_w_rel_error_og,total_w_rel_error_opt
0,1.544739,0.429135


In [19]:
opt_evaluation_path = os.path.join(OUTPUT_DIR_PATH, region, run_id, f'opt_evaluation_{run_id}.xlsx')

with pd.ExcelWriter(opt_evaluation_path) as writer:
    merged_df.to_excel(writer, sheet_name='evaluation_report', index=False)
    total_w_rel_error_df.to_excel(writer, sheet_name='accumulated_error', index=False)